In [ ]:
import random

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy


# ----------------------------------
# State
# ----------------------------------

class State(TypedDict):
    result: str


# ----------------------------------
# Node
# ----------------------------------

def unreliable_node(state: State):

    print("Executing node...")

    # Simulate temporary failure
    if random.random() < 0.7:

        print("Node failed!")

        raise TimeoutError(
            "Temporary service failure"
        )

    print("Node succeeded!")

    return {
        "result": "Data successfully retrieved"
    }


# ----------------------------------
# Build graph
# ----------------------------------

builder = StateGraph(State)


builder.add_node(
    "unreliable",
    unreliable_node,

    retry_policy=RetryPolicy(
        max_attempts=3,
        backoff_factor=4,
        jitter=True,
        retry_on=(RuntimeError, TimeoutError)
    )
)


builder.add_edge(
    START,
    "unreliable"
)

builder.add_edge(
    "unreliable",
    END
)


# ----------------------------------
# Compile
# ----------------------------------

graph = builder.compile()


# ----------------------------------
# Run
# ----------------------------------

result = graph.invoke({
    "result": ""
})


print(
    "Final result:",
    result["result"]
)

Executing node...
Node failed!
Executing node...
Node succeeded!
Final result: Data successfully retrieved
